In [ ]:
import tensorflow as tf
import numpy as np
import json

from matplotlib import pyplot as plt
from IPython.display import display, clear_output
from dataclasses import dataclass, fields
from tensorflow.keras import layers
from pathlib import Path
from PIL import Image

# Data

In [ ]:
def load_images_from_dir():
    ds_path = Path("/kaggle/input/datasets/spandan2/cats-faces-64x64-for-generative-models/cats")

    raw_images = np.stack([
        np.array(Image.open(f).convert("RGB").resize((64, 64)))
        for f in ds_path.iterdir()
        if f.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    ])

    return raw_images

In [ ]:
def preprocess_image(raw_image):
    """casts uint8 [0, 255] -> float32 [-1.0, 1.0]."""

    image = tf.cast(raw_image, tf.float32)
    image = (image / 127.5) - 1.0
    image.set_shape([64, 64, 3])

    return image

In [ ]:
def build_train_dataset(raw_images, global_batch_size):
    train_dataset = (
        tf.data.Dataset.from_tensor_slices(raw_images)
        .shuffle(buffer_size=raw_images.shape[0], reshuffle_each_iteration=True)
        .map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
        .batch(
            global_batch_size,
            drop_remainder=True,
        )
        .repeat()
        .prefetch(tf.data.AUTOTUNE)
    )

    return train_dataset

In [ ]:
def build_dist_train_dataset(strategy, global_batch_size):
    raw_images = load_images_from_dir()
    N_SAMPLES = raw_images.shape[0]

    dataset = build_train_dataset(raw_images, global_batch_size)
    dist_dataset = strategy.experimental_distribute_dataset(dataset)

    return dist_dataset, N_SAMPLES

# Misc

In [ ]:
@dataclass(frozen=True)
class Paths:
    artifact_dir: Path = Path('artifacts')
    checkpoint_dir: Path = artifact_dir / 'checkpoints'
    generated_images_dir: Path = artifact_dir / 'generated_images'
    log_dir: Path = artifact_dir / 'logs'
    metrics_path: Path = artifact_dir / 'metrics.json'


for field in fields(Paths):
    path = getattr(Paths, field.name)

    if path.suffix:
        path.parent.mkdir(parents=True, exist_ok=True)
    else:
        path.mkdir(parents=True, exist_ok=True)

In [ ]:
def get_strategy(strategy_str: str = 'gpu') -> tf.distribute.Strategy:
    strategy_str = strategy_str.lower()

    if strategy_str == 'tpu':
        resolver = tf.distribute.cluster_resolver.TPUClusterResolver()
        tf.config.experimental_connect_to_cluster(resolver)
        tf.tpu.experimental.initialize_tpu_system(resolver)

        return tf.distribute.TPUStrategy(resolver)

    return tf.distribute.MirroredStrategy()

# Models

In [ ]:
# generator for 64x64x3 images
def build_generator(latent_dim=128):
    inputs = layers.Input(shape=(latent_dim,))

    # 4x4 resolution
    x = layers.Dense(4 * 4 * 512, use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Reshape((4, 4, 512))(x)

    # 4x4 -> 8x8
    x = layers.Conv2DTranspose(256, kernel_size=4, strides=2, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)

    # 8x8 -> 16x16
    x = layers.Conv2DTranspose(128, kernel_size=4, strides=2, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)

    # 16x16 -> 32x32
    x = layers.Conv2DTranspose(64, kernel_size=4, strides=2, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)

    # 32x32 -> 64x64
    x = layers.Conv2DTranspose(32, kernel_size=4, strides=2, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)

    # output: (64, 64, 3) normalized in [-1, 1]
    outputs = layers.Conv2D(3, kernel_size=3, padding="same", activation="tanh")(x)

    return tf.keras.Model(inputs, outputs, name="generator")

In [ ]:
def build_critic(image_shape=(64, 64, 3)):
    # note: no BatchNormalization in WGAN-GP critic
    inputs = layers.Input(shape=image_shape)

    # 64x64 -> 32x32
    x = layers.Conv2D(64, kernel_size=4, strides=2, padding="same")(inputs)
    x = layers.LeakyReLU(0.2)(x)

    # 32x32 -> 16x16
    x = layers.Conv2D(128, kernel_size=4, strides=2, padding="same")(x)
    x = layers.LeakyReLU(0.2)(x)

    # 16x16 -> 8x8
    x = layers.Conv2D(256, kernel_size=4, strides=2, padding="same")(x)
    x = layers.LeakyReLU(0.2)(x)

    # 8x8 -> 4x4
    x = layers.Conv2D(512, kernel_size=4, strides=2, padding="same")(x)
    x = layers.LeakyReLU(0.2)(x)

    x = layers.Flatten()(x)
    outputs = layers.Dense(1)(x)

    return tf.keras.Model(inputs, outputs, name="critic")

In [ ]:
class WGANGP:
    def __init__(
        self,
        generator: tf.keras.Model,
        critic: tf.keras.Model,
        generator_optimizer: tf.keras.Optimizer,
        critic_optimizer: tf.keras.Optimizer,
        per_replica_batch_size: int,
        global_batch_size: int,
        latent_dim: int = 128,
        gp_weight: float = 10.0,
    ):
        self.generator = generator
        self.critic = critic

        self.g_optimizer = generator_optimizer
        self.c_optimizer = critic_optimizer

        self.latent_dim = latent_dim
        self.gp_weight = gp_weight

        self.global_batch_size = global_batch_size
        self.per_replica_batch_size = per_replica_batch_size

    def _gradient_penalty(self, real_images, fake_images):
        alpha = tf.random.uniform([self.per_replica_batch_size, 1, 1, 1], 0.0, 1.0, dtype=real_images.dtype)
        interpolated = real_images + alpha * (fake_images - real_images)

        with tf.GradientTape() as gp_tape:
            gp_tape.watch(interpolated)
            pred = self.critic(interpolated, training=True)

        grads = gp_tape.gradient(pred, interpolated)

        # epsilon 1e-12 prevents sqrt(0) -> NaN gradients
        norm = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=[1, 2, 3]) + 1e-12)
        gp_per_sample = tf.square(norm - 1.0)

        return gp_per_sample

    def critic_step(self, real_images):
        noise = tf.random.normal([self.per_replica_batch_size, self.latent_dim], dtype=real_images.dtype)

        with tf.GradientTape() as tape:
            fake_images = self.generator(noise, training=True)
            fake_logits = self.critic(fake_images, training=True)
            real_logits = self.critic(real_images, training=True)

            # per_sample
            wasserstein_loss_per_sample = fake_logits - real_logits
            gp_per_sample = self._gradient_penalty(real_images, fake_images)
            critic_loss_per_sample = wasserstein_loss_per_sample + self.gp_weight * tf.reshape(gp_per_sample, [-1, 1])

            # distributed-safe
            wasserstein_distance = tf.nn.compute_average_loss(-wasserstein_loss_per_sample, global_batch_size=self.global_batch_size)
            gp_loss = tf.nn.compute_average_loss(gp_per_sample, global_batch_size=self.global_batch_size)
            critic_loss = tf.nn.compute_average_loss(critic_loss_per_sample, global_batch_size=self.global_batch_size)

            # equivalent to
            # c_loss = (tf.reduce_sum(fake_logits) - tf.reduce_sum(real_logits) + (self.gp_weight * tf.reduce_sum(gp))) * (1.0 / self.global_batch_size)

        critic_grads = tape.gradient(critic_loss, self.critic.trainable_variables)
        self.c_optimizer.apply_gradients(zip(critic_grads, self.critic.trainable_variables))

        return {
            "wasserstein_distance": wasserstein_distance,
            "gp_loss": gp_loss,
            "critic_loss": critic_loss,
        }

    def generator_step(self):
        noise = tf.random.normal([self.per_replica_batch_size, self.latent_dim])

        with tf.GradientTape() as tape:
            fake_images = self.generator(noise, training=True)
            fake_logits = self.critic(fake_images, training=True)

            gen_loss_per_sample = -fake_logits
            gen_loss = tf.nn.compute_average_loss(gen_loss_per_sample, global_batch_size=self.global_batch_size)

            # equivalent to
            # gen_loss = -tf.reduce_sum(fake_logits) * (1.0 / self.global_batch_size)

        gen_grads = tape.gradient(gen_loss, self.generator.trainable_variables)
        self.g_optimizer.apply_gradients(zip(gen_grads, self.generator.trainable_variables))

        return {
            "generator_loss": gen_loss
        }

In [ ]:
class GANMonitor:
    """
    Handles logging, TensorBoard, checkpoint management and image generation.
    Decoupled completely from GAN math and distributed execution.
    """
    def __init__(
        self,
        wgan: WGANGP,
        latent_dim: int = 128,
        max_checkpoints_to_keep: int = 4,
    ) -> None:
        self.wgan = wgan
        self.writer = tf.summary.create_file_writer(str(Paths.log_dir))

        self.next_epoch = tf.Variable(tf.constant(0, dtype=tf.int64), trainable=False, name="global_epoch")
        self.checkpoint = tf.train.Checkpoint(
            generator=wgan.generator,
            critic=wgan.critic,
            g_optimizer=wgan.g_optimizer,
            c_optimizer=wgan.c_optimizer,
            global_step=self.next_epoch,
        )

        self.manager = tf.train.CheckpointManager(
            self.checkpoint, directory=Paths.checkpoint_dir, max_to_keep=max_checkpoints_to_keep
        )

        self.fixed_noise = tf.random.normal(shape=[12, latent_dim], seed=42)
        self.history = []

    def _log_tensorboard_metrics(self, metrics: dict, step, parent_dir: str, flush: bool = False) -> None:
        with self.writer.as_default():
            for key, val in metrics.items():
                tf.summary.scalar(f"{parent_dir}/{key}", val, step=step)

            if flush:
                self.writer.flush()

    def _log_history(self, metrics: dict, epoch: int) -> None:
        metrics["epoch"] = epoch

        self.history.append(metrics)
        Paths.metrics_path.write_text(json.dumps(self.history, indent=2))

    def _save_checkpoint(self) -> None:
        self.manager.save()

    def _should_log_images(self, epoch: int) -> bool:
        if epoch == 0:
            return True
        elif epoch <= 100:
            interval = (epoch - 1) // 10 + 1
            return epoch % interval == 0
        elif epoch <= 1000:
            return epoch % 25 == 0
        else:
            return epoch % 100 == 0

    def _generate_and_save_images(self, epoch) -> None:
        raw_generated_images = self.wgan.generator(self.fixed_noise, training=False)
        generated_images = (raw_generated_images.numpy() + 1.0) / 2.0  # [-1, 1] -> [0, 1]

        fig, axes = plt.subplots(3, 4, figsize=(8, 6))

        for i, ax in enumerate(axes.flat):
            ax.clear()
            ax.imshow(generated_images[i])
            ax.axis('off')
            # ax.set_title(f"Sample {i+1}", fontsize=8)

        fig.suptitle(f"Epoch {epoch:04d} Fixed Samples", fontsize=12)
        fig.tight_layout()

        fig.savefig(Paths.generated_images_dir / f"epoch_{epoch:04d}.png")

        clear_output(wait=True)
        display(fig)

        plt.close(fig)

        # TensorBoard image logging
        with self.writer.as_default():
            tf.summary.image("fixed_samples", generated_images, max_outputs=12, step=epoch)
            self.writer.flush()


    def restore_checkpoint(self) -> None:
        """Restores checkpoint if available."""

        if self.manager.latest_checkpoint:
            self.checkpoint.restore(self.manager.latest_checkpoint)

            print(f"[Monitor] Restored from {self.manager.latest_checkpoint} at epoch {self.next_epoch.numpy()}")
        else:
            print("[Monitor] No checkpoint found. Starting from scratch.")

    def on_epoch_end(self, metrics: dict, epoch: int) -> None:
        """Handles metric recording, saving plots and checkpointing at epoch completion."""

        self._save_checkpoint()
        self._log_tensorboard_metrics(metrics, epoch, "epoch", flush=True)
        self._log_history(metrics, epoch)

        if self._should_log_images(epoch):
            self._generate_and_save_images(epoch)

        self.next_epoch.assign_add(1)

    def on_step_end(self, metrics: dict, step: int) -> None:
        if step % 5 == 0:
            self._log_tensorboard_metrics(metrics, step, "step")

In [ ]:
class DistributedWGANGPTrainer:
    def __init__(
        self,
        strategy: tf.distribute.Strategy,
        wgan: WGANGP,
        monitor: GANMonitor,
        per_replica_batch_size: int,
        global_batch_size: int,
        n_critic_steps: int = 5,
    ):
        self.strategy = strategy

        self.wgan = wgan
        self.monitor = monitor

        self.n_critic_steps = n_critic_steps

        self.per_replica_batch_size = per_replica_batch_size
        self.global_batch_size = global_batch_size

    def _reduce_per_replica_metrics(self, metrics: dict[str, tf.Tensor]) -> dict[str, tf.Tensor]:
        return {k: self.strategy.reduce(tf.distribute.ReduceOp.SUM, v, axis=None) for k, v in metrics.items()}

    @tf.function
    def _distributed_critic_step(self, real_images) -> dict[str, tf.Tensor]:
        per_replica_metrics = self.strategy.run(self.wgan.critic_step, args=(real_images,))
        metrics = self._reduce_per_replica_metrics(per_replica_metrics)

        return metrics

    @tf.function
    def _distributed_generator_step(self) -> dict[str, tf.Tensor]:
        per_replica_metrics = self.strategy.run(self.wgan.generator_step)
        metrics = self._reduce_per_replica_metrics(per_replica_metrics)

        return metrics

    def train(self, distributed_dataset, total_samples: int, epochs: int):
        data_iter = iter(distributed_dataset)

        total_batches = total_samples // self.global_batch_size
        steps_per_epoch = total_batches // self.n_critic_steps

        for epoch in range(self.monitor.next_epoch.numpy(), epochs):
            epoch_metrics = {}

            for step in range(steps_per_epoch):
                for _ in range(self.n_critic_steps):
                    real_images = next(data_iter)
                    critic_metrics = self._distributed_critic_step(real_images)

                gen_metrics = self._distributed_generator_step()
                step_metrics = critic_metrics | gen_metrics

                for k, v in step_metrics.items():
                    epoch_metrics.setdefault(k, []).append(v)

                self.monitor.on_step_end(step_metrics, step)

                if step % 5 == 0:
                    print(
                        f"Epoch [{epoch + 1}/{epochs}] "
                        f"Step [{step + 1}/{steps_per_epoch}] "
                        + " | ".join(f"{k}: {v:.4f}" for k, v in step_metrics.items())
                    )

            epoch_metrics = {
                k: float(tf.reduce_mean(tf.stack(v))) for k, v in epoch_metrics.items()
            }

            self.monitor.on_epoch_end(epoch_metrics, epoch)

# Training

In [ ]:
strategy = get_strategy('gpu')
print(f"Number of synchronized devices: {strategy.num_replicas_in_sync}")

In [ ]:
PER_REPLICA_BATCH_SIZE = 64
GLOBAL_BATCH_SIZE = strategy.num_replicas_in_sync * PER_REPLICA_BATCH_SIZE

LATENT_DIM = 128
GP_WEIGHT = 10.0
N_CRITIC_STEPS = 5

EPOCHS = 1001

In [ ]:
dist_dataset, N_SAMPLES = build_dist_train_dataset(strategy, GLOBAL_BATCH_SIZE)

In [ ]:
with strategy.scope():
    generator = build_generator(latent_dim=LATENT_DIM)
    critic = build_critic(image_shape=(64, 64, 3))

    generator_optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4, beta_1=0.0, beta_2=0.9)
    critic_optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4, beta_1=0.0, beta_2=0.9)

    # pre-allocates variables across all replicas in eager mode
    # this prevents lazy tf.cond initialization inside the graph
    generator_optimizer.build(generator.trainable_variables)
    critic_optimizer.build(critic.trainable_variables)

    wgan = WGANGP(
        generator=generator,
        critic=critic,
        generator_optimizer=generator_optimizer,
        critic_optimizer=critic_optimizer,
        per_replica_batch_size=PER_REPLICA_BATCH_SIZE,
        global_batch_size=GLOBAL_BATCH_SIZE,
        latent_dim=LATENT_DIM,
        gp_weight=GP_WEIGHT,
    )

monitor = GANMonitor(
    wgan=wgan,
    latent_dim=LATENT_DIM,
)

# monitor.restore_checkpoint()

trainer = DistributedWGANGPTrainer(
    strategy=strategy,
    wgan=wgan,
    monitor=monitor,
    per_replica_batch_size=PER_REPLICA_BATCH_SIZE,
    global_batch_size=GLOBAL_BATCH_SIZE,
    n_critic_steps=N_CRITIC_STEPS,
)

print("Starting multi-GPU training...")
trainer.train(dist_dataset, total_samples=N_SAMPLES, epochs=EPOCHS)